# Loan Default Prediction
Use this notebook to load the saved trained model, select an applicant CSV, and display default probabilities and risk bands.

**Prerequisite:** Run the training workflow first so `artifacts/loan_default_model.joblib` exists.

This notebook preserves the uploaded `predict.py` workflow. It displays results in the notebook and does not save a prediction CSV.

## 1. Import libraries

In [1]:
from pathlib import Path
import joblib
import pandas as pd
from IPython.display import display, Markdown
import tkinter as tk
from tkinter import filedialog, messagebox

## 2. Locate the saved model
By default, this checks for the `artifacts` folder in the current working directory. If needed, update `PROJECT_DIR` to your project folder.

In [2]:
PROJECT_DIR = Path.cwd()
MODEL_PATH = PROJECT_DIR / 'artifacts' / 'loan_default_model.joblib'
print(f'Project folder: {PROJECT_DIR.resolve()}')
print(f'Model path: {MODEL_PATH.resolve()}')
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Saved model not found: {MODEL_PATH}\n'
        'Run the training notebook/script first, or set PROJECT_DIR correctly.'
    )

Project folder: C:\Users\abhis\Downloads\loan_default_prediction_project\loan_default_prediction_project
Model path: C:\Users\abhis\Downloads\loan_default_prediction_project\loan_default_prediction_project\artifacts\loan_default_model.joblib


## 3. Select the applicant CSV file

In [3]:
root = tk.Tk()
root.withdraw()
try:
    root.attributes('-topmost', True)
except tk.TclError:
    pass

try:
    input_file = filedialog.askopenfilename(
        title='Select Applicant CSV File',
        initialdir=str(PROJECT_DIR),
        filetypes=[('CSV files', '*.csv'), ('All files', '*.*')],
    )
finally:
    root.destroy()

if not input_file:
    raise RuntimeError('No file selected. Prediction cancelled.')
print(f'Selected file: {input_file}')

Selected file: C:/Users/abhis/Downloads/applicant_data.csv


## 4. Load model and applicant data

In [4]:
bundle = joblib.load(MODEL_PATH)
model = bundle['model']
model_name = bundle['model_name']
feature_columns = bundle['feature_columns']
categorical_columns = bundle.get('categorical_columns', [])
threshold = float(bundle.get('threshold', 0.5))

df = pd.read_csv(input_file, low_memory=False)
df.columns = [str(col).strip() for col in df.columns]
if df.empty:
    raise ValueError('The selected CSV contains no applicant records.')

missing_columns = [col for col in feature_columns if col not in df.columns]
if missing_columns:
    raise ValueError(
        'The selected CSV is missing required feature columns: '
        + ', '.join(missing_columns)
        + '\nUse the same input features as the training dataset.'
    )

X = df[feature_columns].copy()
if model_name == 'CatBoost':
    for col in categorical_columns:
        if col in X.columns:
            X[col] = X[col].astype('string').fillna('__MISSING__').astype(str)

print(f'Model: {model_name}')
print(f'Applicants loaded: {len(df)}')

Model: Random Forest
Applicants loaded: 5


## 5. Generate predictions

In [5]:
probabilities = model.predict_proba(X)[:, 1]
predictions = probabilities >= threshold

results = pd.DataFrame({
    'Applicant': range(1, len(df) + 1),
    'Default Probability (%)': (probabilities * 100).round(2),
    'Predicted Default': ['Yes' if value else 'No' for value in predictions],
    'Risk Band': pd.cut(
        probabilities,
        bins=[-0.001, 0.20, 0.50, 1.001],
        labels=['Low', 'Medium', 'High'],
        include_lowest=True,
    ).astype(str),
})
display(Markdown(f'### Loan Default Prediction Results\n\n'
                 f'- **Model:** {model_name}\n'
                 f'- **Default threshold:** {threshold:.2f}\n'
                 f'- **Applicants processed:** {len(results)}'))
display(results)

### Loan Default Prediction Results

- **Model:** Random Forest
- **Default threshold:** 0.50
- **Applicants processed:** 5

,Applicant,Default Probability (%),Predicted Default,Risk Band
0,1,38.82,No,Medium
1,2,55.72,Yes,High
2,3,26.17,No,Medium
3,4,62.41,Yes,High
4,5,40.76,No,Medium


## 6. Prediction summary

In [6]:
display(Markdown('### Predicted default counts'))
display(results['Predicted Default'].value_counts().rename_axis('Prediction').to_frame('Count'))
display(Markdown('### Risk band counts'))
display(results['Risk Band'].value_counts().rename_axis('Risk Band').to_frame('Count'))
print('Predictions are displayed in the notebook; no CSV was saved.')

### Predicted default counts

,Count
Prediction,
No,3
Yes,2


### Risk band counts

,Count
Risk Band,
Medium,3
High,2


Predictions are displayed in the notebook; no CSV was saved.


## Notes
- The applicant CSV must contain every feature column used during training.
- Do not use the target column `Default` as a substitute for a missing feature.
- This notebook does not train a model; it loads the saved model from `artifacts/`.
- Treat the output as an educational prediction, not a standalone lending decision.